In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import joblib
import os

# --- Utility: safe division to avoid divide-by-zero ---
def safe_divide(numer, denom, eps=1e-8):
    return numer / np.maximum(np.abs(denom), eps)

# --- Robust MAPE (no inf) ---
def mean_absolute_percentage_error_safe(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs(safe_divide(y_true - y_pred, y_true, eps))) * 100

# --- Accuracy helpers using safe_divide ---
def calculate_percentage_accuracy(y_true, y_pred, tolerance_percent=5, eps=1e-8):
    tol = tolerance_percent / 100.0
    rel_err = np.abs(safe_divide(y_true - y_pred, y_true, eps))
    return np.mean(rel_err <= tol) * 100

def calculate_threshold_accuracy(y_true, y_pred, threshold=0.05):
    return np.mean(np.abs(y_true - y_pred) <= threshold) * 100

# --- Binned accuracy (keeps your original idea) ---
def calculate_binned_accuracy(y_true, y_pred, n_bins=5):
    bins = np.linspace(y_true.min(), y_true.max(), n_bins + 1)
    y_true_b = np.digitize(y_true, bins) - 1
    y_pred_b = np.digitize(y_pred, bins) - 1
    y_true_b = np.clip(y_true_b, 0, n_bins - 1)
    y_pred_b = np.clip(y_pred_b, 0, n_bins - 1)
    return accuracy_score(y_true_b, y_pred_b) * 100, bins

# -------------------------
# LOAD
# -------------------------
splits_dir = '../DATA/splits'
X_train = pd.read_csv(f'{splits_dir}/X_train.csv')
X_val   = pd.read_csv(f'{splits_dir}/X_val.csv')
X_test  = pd.read_csv(f'{splits_dir}/X_test.csv')
y_train = pd.read_csv(f'{splits_dir}/y_train.csv').iloc[:, 0]
y_val   = pd.read_csv(f'{splits_dir}/y_val.csv').iloc[:, 0]
y_test  = pd.read_csv(f'{splits_dir}/y_test.csv').iloc[:, 0]

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# -------------------------
# AUTO-DETECT VERY HIGH CORR (possible leakage)
# -------------------------
df_train = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True).rename('target')], axis=1)
corr_with_target = df_train.corr()['target'].drop('target').abs()
leakage_thresh = 0.95
suspected_leak = corr_with_target[corr_with_target >= leakage_thresh].index.tolist()

if suspected_leak:
    print("⚠️ Suspected leakage/highly-correlated features (>= {:.2f}): {}".format(leakage_thresh, suspected_leak))
    # drop them from all sets
    X_train = X_train.drop(columns=suspected_leak, errors='ignore')
    X_val   = X_val.drop(columns=suspected_leak, errors='ignore')
    X_test  = X_test.drop(columns=suspected_leak, errors='ignore')
else:
    print("No extremely-highly correlated features detected (>= {:.2f})".format(leakage_thresh))

# -------------------------
# SCALE FEATURES
# -------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# -------------------------
# TRAIN XGBoost
# -------------------------
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=30
)

eval_set = [(X_train_scaled, y_train), (X_val_scaled, y_val)]
xgb_model.fit(X_train_scaled, y_train, eval_set=eval_set, verbose=False)

print("Trained. best_iteration:", getattr(xgb_model, "best_iteration", None))

# -------------------------
# PREDICT
# -------------------------
y_train_pred = xgb_model.predict(X_train_scaled)
y_val_pred   = xgb_model.predict(X_val_scaled)
y_test_pred  = xgb_model.predict(X_test_scaled)

# -------------------------
# EVALUATE (robust RMSE, safe MAPE)
# -------------------------
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)                      # <-- compatible across sklearn versions
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error_safe(y_true, y_pred)
    acc5 = calculate_percentage_accuracy(y_true, y_pred, 5)
    acc10 = calculate_percentage_accuracy(y_true, y_pred, 10)
    thr005 = calculate_threshold_accuracy(y_true, y_pred, 0.05)
    thr01 = calculate_threshold_accuracy(y_true, y_pred, 0.10)
    bacc, bins = calculate_binned_accuracy(y_true, y_pred, n_bins=5)

    print(f"\n{name} metrics:")
    print(f" MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}  MAPE_safe: {mape:.2f}%")
    print(f" Accuracy within 5%: {acc5:.2f}%, within 10%: {acc10:.2f}%")
    print(f" Threshold abs<=0.05: {thr005:.2f}%, abs<=0.10: {thr01:.2f}%")
    print(f" Binned(5) accuracy: {bacc:.2f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape, "Acc5": acc5, "Acc10": acc10, "Binned": bacc}

train_metrics = evaluate(y_train.values, y_train_pred, "TRAIN")
val_metrics   = evaluate(y_val.values, y_val_pred, "VALIDATION")
test_metrics  = evaluate(y_test.values, y_test_pred, "TEST")

# -------------------------
# Overfitting quick check
# -------------------------
print("\nOverfitting check - RMSE (Train, Val, Test):", train_metrics['RMSE'], val_metrics['RMSE'], test_metrics['RMSE'])
if val_metrics['RMSE'] - train_metrics['RMSE'] > 0.05:
    print("⚠️ Possible overfitting (Val RMSE much larger than Train RMSE).")
else:
    print("No major overfitting sign based on RMSE diff.")

# -------------------------
# Save artifacts
# -------------------------
models_dir = '../models_fixed'
os.makedirs(models_dir, exist_ok=True)
xgb_model.save_model(f'{models_dir}/xgb_fixed.json')
joblib.dump(scaler, f'{models_dir}/feature_scaler.pkl')
print("Saved model & scaler in", models_dir)


Shapes: (10544, 11) (2260, 11) (2260, 11)
⚠️ Suspected leakage/highly-correlated features (>= 0.95): ['Rolling_SOH']
Trained. best_iteration: 499

TRAIN metrics:
 MAE: 0.0051  RMSE: 0.0070  R2: 0.9988  MAPE_safe: 0.88%
 Accuracy within 5%: 99.27%, within 10%: 99.97%
 Threshold abs<=0.05: 99.98%, abs<=0.10: 100.00%
 Binned(5) accuracy: 96.45%

VALIDATION metrics:
 MAE: 0.1039  RMSE: 0.1152  R2: -5.5126  MAPE_safe: 52.69%
 Accuracy within 5%: 2.70%, within 10%: 5.97%
 Threshold abs<=0.05: 13.54%, abs<=0.10: 45.04%
 Binned(5) accuracy: 9.65%

TEST metrics:
 MAE: 0.2356  RMSE: 0.2435  R2: -32.1377  MAPE_safe: 44216877.19%
 Accuracy within 5%: 0.00%, within 10%: 0.00%
 Threshold abs<=0.05: 0.00%, abs<=0.10: 0.00%
 Binned(5) accuracy: 8.54%

Overfitting check - RMSE (Train, Val, Test): 0.007005004298101459 0.11521928703775008 0.24350722703681824
⚠️ Possible overfitting (Val RMSE much larger than Train RMSE).
Saved model & scaler in ../models_fixed
